In [1]:
import numpy as np
import pandas as pd
with open('matrix.npy', 'rb') as f:
    cosine_sim = np.load(f)

In [2]:
indices = pd.read_csv('indices.csv')

In [35]:
print(json.__version__)

2.0.9


In [23]:
def get_recommendations(title, a_list, cosine_sim=cosine_sim, num_recommend = 10):
    if title in indices['title_romaji'].values:
        num_recd = 0
        raw_recd = 1
        top_similar = []
        idx = (indices['title_romaji'] == title).idxmax()
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        while num_recd <= num_recommend:
            top_rec = sim_scores[raw_recd]
            if indices['title_romaji'].iloc[top_rec[0]] not in a_list and "Season" not in indices['title_romaji'].iloc[top_rec[0]]:
                top_similar.append(sim_scores[raw_recd])
                num_recd += 1
            raw_recd += 1
        movie_indices = [i[0] for i in top_similar]
        return indices['title_romaji'].iloc[movie_indices]

In [4]:
import requests
import json
query = '''
query ($type: MediaType!, $userName: String!) {
  MediaListCollection(type: $type, userName: $userName) {
    lists {
      name
      entries {
        id
        media {
          title {
            romaji
          }
        }
      }
    }
  }
}
'''

variables = {
    'type': 'ANIME',
    'userName':'aryantestlist'
}

url = 'https://graphql.anilist.co'

response = requests.post(url, json={'query': query, 'variables': variables})

In [7]:
json_data = json.loads(response.content)
anime_list = []
for j in json_data['data']['MediaListCollection']['lists']:
    if j['name'] != "Dropped" and j['name'] != "Planning":
        for show in j['entries']:
            anime_list.append(str(show['media']['title']['romaji']))
for show in anime_list:
    if show not in indices['title_romaji'].values:
        print(show)
        anime_list.remove(show)

Jujutsu Kaisen: Shimetsu Kaiyuu - Zenpen
Shoushimin Series 2nd Season
Kill Ao


In [24]:
total_recs = {}
for title in anime_list:  
    recs = list(get_recommendations(title, anime_list, num_recommend = 10))
    for rec in recs:
        if rec not in anime_list:
            if rec in total_recs:
                total_recs[rec] += recs.index(rec)
            else:
                total_recs[rec] = recs.index(rec)

In [25]:
dict(sorted(total_recs.items(), key=lambda item: item[1], reverse=True))

{'Tokyo Revengers: Tenjiku-hen': 54,
 'VINLAND SAGA': 38,
 'Bakuten!!': 35,
 'Shinsekai yori': 33,
 'Hagane no Renkinjutsushi: FULLMETAL ALCHEMIST': 32,
 'Code Geass: Hangyaku no Lelouch R2': 30,
 'Summer Time Render': 27,
 'Higurashi no Naku Koro ni Gou': 27,
 'Free!': 26,
 'Enen no Shouboutai: San no Shou': 26,
 'Tougen Anki': 25,
 'PLUTO': 25,
 'CLANNAD': 25,
 'ONE PIECE': 24,
 'Tokyo Revengers': 23,
 'Yuuki Yuuna wa Yuusha de Aru': 22,
 'Sentai Daishikkaku': 22,
 'Ima, Soko ni Iru Boku': 21,
 'Ginga Eiyuu Densetsu': 20,
 'Shakunetsu Kabaddi': 19,
 'Kuroko no Basket 2nd SEASON': 19,
 'Eyeshield 21': 19,
 'Gachiakuta': 19,
 'Shin Seiki Evangelion Movie: Air / Magokoro wo, Kimi ni': 19,
 'Haikyuu!! Riku VS Kuu': 18,
 'Katekyou Hitman REBORN!': 18,
 'SANDA': 18,
 'Kimetsu no Yaiba: Mugenjou-hen Movie 1 - Akaza Sairai': 18,
 'Magi: The kingdom of magic': 18,
 'Inuyashiki': 17,
 'Overlord III': 17,
 'Go-toubun no Hanayome Movie': 17,
 'Diamond no Ace': 16,
 'Sakigake!! Cromartie Koukou':